# gpumemd — benchmark Colab / NVIDIA T4
In Colab scegli **Runtime → Change runtime type → T4 GPU**, poi esegui le celle in ordine.
Carica `gpumemd-colab-source.zip` quando richiesto: contiene il codice locale da misurare.
Scarica lo ZIP dalla [release v0.18.1](https://github.com/yahiag04/gpumemd/releases/tag/v0.18.1).
Usa una sessione nuova, senza altri workload GPU. Non importa PyTorch/TensorFlow.

Misura memoria e caricamento di byte sintetici, **non inferenza**. Il file è scritto su disco locale;
le cache del filesystem non vengono svuotate. Cold significa modello non residente in GPU.
La disponibilità della T4 dipende da Colab: https://research.google.com/colaboratory/faq.html
Architettura T4: compute capability 7.5, https://developer.nvidia.com/cuda/gpus


In [ ]:
from pathlib import Path
import csv, json, os, shutil, subprocess, sys, tempfile, time
from google.colab import files

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
ROOT = Path('/content/gpumemd')
RESULTS = Path(tempfile.mkdtemp(prefix='gpumemd-results-', dir='/content'))
def run(args, log=None, timeout=1800):
    p = subprocess.run(list(map(str, args)), text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout)
    if log:
        (RESULTS / log).write_text(p.stdout)
    print(p.stdout)
    p.check_returncode()
    return p.stdout

gpu = run(['nvidia-smi', '-i', '0', '--query-gpu=name,driver_version,memory.total,memory.free',
           '--format=csv'], 'gpu.csv')
assert 'T4' in gpu, 'Seleziona un runtime T4 e riprova.'
nvcc = shutil.which('nvcc')
assert nvcc, 'nvcc assente: serve un runtime Colab con CUDA toolkit.'
run([nvcc, '--version'], 'nvcc.txt')
if not ROOT.exists():
    uploaded = files.upload()  # scegli gpumemd-colab-source.zip
    archive = 'gpumemd-colab-source.zip'
    assert archive in uploaded, f'Carica {archive}'
    ROOT.mkdir()
    shutil.unpack_archive(archive, ROOT)
    del uploaded
assert (ROOT / 'scripts/benchmark.py').is_file()
if (ROOT / 'SOURCE_REVISION.txt').exists():
    shutil.copy(ROOT / 'SOURCE_REVISION.txt', RESULTS)
run([sys.executable, '-m', 'pip', 'install', '-q', 'cmake'])
BUILD = ROOT / 'build-colab'
run(['cmake', '-S', ROOT, '-B', BUILD, '-DCMAKE_BUILD_TYPE=Release',
     '-DCMAKE_CUDA_ARCHITECTURES=75', f'-DCMAKE_CUDA_COMPILER={nvcc}'], 'configure.txt')
run(['cmake', '--build', BUILD, '--parallel', '2'], 'build.txt')
assert (BUILD / 'gpumemd-cuda-bench-worker').is_file(), 'Target CUDA mancante'
run(['ctest', '--test-dir', BUILD, '--output-on-failure'], 'ctest.txt')


## Parametri
512 MiB × 8 worker = 4 GiB di payload indipendente, più contesti CUDA e overhead.
Tre ripetizioni per N; aumenta `TRIALS` per studiare la variabilità.
I percentili cold con 20 campioni sono indicativi; p99 è sostanzialmente il massimo.


In [ ]:
SIZE_MIB = 512
PROCESS_COUNTS = [1, 2, 4, 8]
TRIALS = 3
COLD_ITERATIONS = 20
MODEL = ROOT / 'colab-model.bin'
assert SIZE_MIB > 0 and TRIALS > 0 and all(n > 0 for n in PROCESS_COUNTS)
with MODEL.open('wb') as f:
    block = bytes(range(256)) * 4096  # 1 MiB, realmente scritto
    for _ in range(SIZE_MIB):
        f.write(block)
(RESULTS / 'parameters.json').write_text(json.dumps(dict(
    size_mib=SIZE_MIB, processes=PROCESS_COUNTS, trials=TRIALS,
    cold_iterations=COLD_ITERATIONS), indent=2))
run([BUILD / 'gpumemd-bench', '--file', MODEL, '--backend', 'cuda',
     '--iterations', COLD_ITERATIONS], 'cold-warm.txt')


## Socket, CUDA IPC e consumo VRAM
Il demo verifica IPC tra processi. Poi ogni prova confronta N copie indipendenti con
una copia condivisa, usando lo stesso file. I contesti worker sono inizializzati prima
del baseline; i delta GPU-wide non rappresentano la VRAM totale dell'applicazione.
L'ordine è sempre independent → shared: cache e clock possono influenzare i tempi.
La sincronizzazione misura una barriera su device inattivo.


In [ ]:
work = Path(tempfile.mkdtemp(prefix='gpumemd-', dir='/tmp'))
sock = work / 'broker.sock'
base = [sys.executable, ROOT / 'scripts/benchmark.py', '--socket', sock, '--model', 'bench']
def ctl(*args):
    return subprocess.check_output([str(BUILD / 'gpumemctl'), '--socket', str(sock),
                                    *map(str, args)], text=True)
rows = []
with (RESULTS / 'daemon.log').open('w') as log:
    daemon = subprocess.Popen([str(BUILD / 'gpumemd'), '--memory', str(MODEL.stat().st_size * 2),
                               '--socket', str(sock)], stdout=log, stderr=log)
    try:
        for _ in range(100):
            if sock.exists():
                break
            if daemon.poll() is not None:
                raise RuntimeError((RESULTS / 'daemon.log').read_text())
            time.sleep(0.1)
        else:
            raise TimeoutError('Daemon non pronto')
        print(ctl('register_file', 'bench', MODEL, 'colab'))
        print(ctl('load', 'bench'))
        run([BUILD / 'gpumemd-cuda-demo', '--socket', sock, '--model', 'bench'], 'ipc.txt')
        run(base + ['--samples', 10000], 'socket.json')
        for n in PROCESS_COUNTS:
            for trial in range(1, TRIALS + 1):
                output = run(base + ['--cuda-worker', BUILD / 'gpumemd-cuda-bench-worker',
                                    '--file', MODEL, '--processes', n], f'memory-n{n}-trial{trial}.txt')
                measurements = [json.loads(line) for line in output.splitlines() if line.startswith('{')]
                assert [m['mode'] for m in measurements] == ['independent', 'shared']
                independent, shared = [m['delta_mib'] for m in measurements]
                valid = independent > 0 and shared > 0
                rows.append(dict(processes=n, trial=trial, independent_mib=independent,
                                 shared_mib=shared, positive_deltas=valid,
                                 savings_percent=100 * (1 - shared / independent) if valid else None))
        (RESULTS / 'metrics.txt').write_text(ctl('metrics'))
        print(ctl('status'))
    finally:
        daemon.terminate()
        try:
            daemon.wait(timeout=10)
        except subprocess.TimeoutExpired:
            daemon.kill()
            daemon.wait()
        shutil.rmtree(work)


## Acquire/release: prenotazioni logiche
Daemon separato: verifica acquisizione, duplicati, capacità, timeout, attesa sbloccata dal rilascio e contatori bilanciati. Non misura allocazioni CUDA.


In [ ]:
run([sys.executable, ROOT / 'scripts/check_acquire_release.py', '--build', BUILD], 'acquire-release.json')


## Risultati ed esportazione
`memory.csv`: VRAM incrementale per prova e risparmio `100 × (1 − shared / independent)`.
Solo per il payload ideale, N=1/2/4/8 corrisponderebbe a 0/50/75/87,5%: **non sono risultati misurati**.
Delta non positivi sono invalidi; anche delta positivi possono essere rumorosi: confronta le ripetizioni.
`cold-warm.txt`: tempi medi e p50/p95/p99 in-process; `socket.json`: p50/p95/p99 round-trip
(connessione e overhead Python inclusi). I log delle prove includono caricamento/apertura IPC,
sync, chiusura e caricamento broker separato: il solo tempo di apertura IPC esclude quest'ultimo.
Scarica lo ZIP prima della disconnessione. Se una cella fallisce, esegui comunque questa cella
per scaricare i log diagnostici disponibili.


In [ ]:
if 'rows' in globals() and rows:
    with (RESULTS / 'memory.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    import pandas as pd
    display(pd.DataFrame(rows))
archive = shutil.make_archive(str(RESULTS), 'zip', RESULTS)
files.download(archive)
